# Bitwise Operators — Why, How and Practice

**Software Programming (IGS1931) · Week 7**

In Lecture 09 you met operators like `&`, `|`, `^`, `<<` and `>>`. They look strange at first, so today we start with the question **"why would anyone use these?"**

1. **Part 1:** bits in 3 minutes, as a warm-up
2. **Part 2:** 7 real examples (permission systems, data filtering with **pandas**, and colors). Each example also explains **how** its operator works.
3. **Part 3:** all the operators together in one place, plus assignment operators
4. **Part 4:** a few quick loop examples (`for` / `while`)
5. **Part 5:** 10 exercises

Run the cells **in order**, because some later cells use things defined earlier.

## Today You Will Be Able To...

- Explain **why** bitwise operators exist, using real examples
- Store several on/off settings in **one number**, then check, add, flip and remove them
- Filter a pandas table with `&`, `|`, `~` and `^`, and avoid the two classic mistakes
- Work out `&`, `|`, `^`, `~`, `<<` and `>>` by hand, column by column
- Convert between decimal, binary (`0b`), hexadecimal (`0x`) and octal (`0o`)
- Use assignment operators such as `|=`, `^=`, `&=` and `>>=`

---
## Part 1 — Bits in 3 Minutes

A computer stores every number as **bits** (0s and 1s). Binary works like normal numbers, except that each position is worth **2×** the one to its right, not 10×:

| place value | 8 | 4 | 2 | 1 |
|---|---|---|---|---|
| `13` in binary | 1 | 1 | 0 | 1 |

`1101` = 8 + 4 + 1 = **13**

In Python:
- `0b1101` writes a number **in binary**. Python prints it as `13`.
- `bin(13)` shows a number in binary as `'0b1101'`.
- `'{:04b}'.format(13)` shows it in binary with exactly 4 digits: `'1101'`.

**👉 Run the cell below first.** It defines `show()`, a helper that prints numbers in binary, lined up in columns. We use it all through today's notebook.

In [ ]:
def show(*numbers, width=4):
    """Print each number in decimal and in binary, lined up in columns."""
    for n in numbers:
        print('{:5d} = {}'.format(n, format(n, '0{}b'.format(width))))


print(0b1101, bin(13))
show(1, 2, 4, 8, 13)

Notice that **1, 2, 4 and 8 each have exactly one `1` bit**. That fact is the key to everything in Part 2.

---
## Part 2 — Where Are Bitwise Operators Used? (7 Examples)

Think of a number as a row of **light switches**, where each bit is one switch: `1` means on and `0` means off. Bitwise operators let you work with **one switch** without touching the others.

### Example 1 — Permission System: give permissions with `|` (OR)

A file can be **read**, **written** or **executed**, and every user has their own permissions. Instead of three separate `True/False` variables, we give each permission **its own bit**:

| permission | value | binary |
|---|---|---|
| `READ_PERMISSION` | 4 | `100` |
| `WRITE_PERMISSION` | 2 | `010` |
| `EXECUTE_PERMISSION` | 1 | `001` |

To give a user read **and** write permission, we **combine** the two with `|`:

In [ ]:
READ_PERMISSION = 4
WRITE_PERMISSION = 2
EXECUTE_PERMISSION = 1

user_permissions = READ_PERMISSION | WRITE_PERMISSION

print(user_permissions)
print(bin(user_permissions))

**How `|` (OR) works:** compare the two numbers **column by column**. The result has a `1` wherever **at least one** of them has a `1`.

| a | b | `a \| b` |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

```
  READ   = 100   (4)
  WRITE  = 010   (2)
  ------------
  a | b  = 110   (6)   ← both switches on
```

This is exactly how Linux stores file permissions. In `chmod 755`, the `7` is `111` (read + write + execute) and the `5` is `101` (read + execute).

### Example 2 — Permission System: check a permission with `&` (AND)

The user has permissions `6`. Can they **read**? Can they **execute**?

In [ ]:
user_permissions = 6

if (user_permissions & READ_PERMISSION) == READ_PERMISSION:
    print("Can Read")
else:
    print("Cannot Read")

if (user_permissions & EXECUTE_PERMISSION) == EXECUTE_PERMISSION:
    print("Can Execute")
else:
    print("Cannot Execute")

print(user_permissions & READ_PERMISSION)
print(user_permissions & EXECUTE_PERMISSION)

**How `&` (AND) works:** the result has a `1` only where **both** numbers have a `1`.

| a | b | `a & b` |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

```
  user     = 110   (6)              user     = 110   (6)
  READ     = 100   (4)              EXECUTE  = 001   (1)
  ------------------                ------------------
  &        = 100   (4) → yes!       &        = 000   (0) → no
```

`READ_PERMISSION` works like a **stencil** (programmers call it a **mask**). It covers every bit except the one we care about. If the result equals the mask, that switch is on.

### Example 3 — Permission System: flip with `^=` (XOR), remove with `&= ~`

Now we want to take away write permission:

In [ ]:
user_permissions = READ_PERMISSION | WRITE_PERMISSION    # 6 = 110

user_permissions ^= WRITE_PERMISSION
print(user_permissions, bin(user_permissions))

user_permissions ^= WRITE_PERMISSION                     # run the same line again...
print(user_permissions, bin(user_permissions))

**How `^` (XOR, "exclusive or") works:** the result has a `1` where the two bits are **different**.

| a | b | `a ^ b` |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

```
  user   = 110   (6)
  WRITE  = 010   (2)
  ------------
  ^      = 100   (4)   ← the write switch was FLIPPED off
```

⚠️ XOR **flips** a switch, so doing it twice turns write permission back **on**. That's useful for a toggle button, such as mute/unmute, but risky if you really want to *remove* a permission.

To **always turn a switch off**, use `&= ~mask`. Here `~` (NOT) flips **every** bit of the mask, so `~WRITE` has the write bit **off** and all other bits **on**. AND-ing with it keeps everything except write:

In [ ]:
user_permissions = READ_PERMISSION | WRITE_PERMISSION    # 110

user_permissions &= ~WRITE_PERMISSION
print(user_permissions, bin(user_permissions))

user_permissions &= ~WRITE_PERMISSION                    # again: still off, safe
print(user_permissions, bin(user_permissions))

**Summary: the permission toolkit**

| goal | code |
|---|---|
| give a permission | `p \|= FLAG` |
| check a permission | `(p & FLAG) == FLAG` |
| flip a permission | `p ^= FLAG` |
| remove a permission | `p &= ~FLAG` |

*(Side note: Python shows `~2` as `-3`, because `~x` always equals `-x - 1`. You don't need to worry about why. Just use it inside `& ~FLAG`.)*

### Example 4 — pandas: filter rows with `&`

**pandas** is the most popular Python library for working with tables of data. It uses the bitwise operators `&`, `|`, `~` and `^` to **combine conditions**.

Here is a small table of students:

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name":       ["Aziz", "Bona", "Chen", "Dilnoza", "Eun", "Farrukh"],
    "score":      [85, 52, 91, 67, 74, 45],
    "attendance": [95, 80, 60, 90, 85, 70],
    "hw1":        [True, True, False, True, False, False],
    "hw2":        [True, False, False, True, True, True],
})

df

A comparison on a column gives a column of `True`/`False` values, one per row. Think of it as **a column of switches**:

In [ ]:
passed = df["score"] >= 60
good_attendance = df["attendance"] >= 80

pd.DataFrame({"name": df["name"], "passed": passed,
              "good_attendance": good_attendance,
              "passed & good_attendance": passed & good_attendance})

`&` compares the two columns **row by row**, just as it compares bits **column by column**. `True` behaves like `1` and `False` like `0`. Use that combined column inside `df[...]` to keep only the matching rows:

In [ ]:
df[(df["score"] >= 60) & (df["attendance"] >= 80)]

⚠️ **Two classic mistakes**

1. **Using `and` instead of `&`.** `and` only works on a **single** True/False, not on a whole column, so pandas raises an error:
   `ValueError: The truth value of a Series is ambiguous.`
2. **Forgetting the parentheses.** `&` runs **before** `>=`, so `df["score"] >= 60 & df["attendance"] >= 80` gets computed in the wrong order. Always wrap each condition in `( )`.

In [ ]:
# Uncomment the line below and run it to see mistake 1 for yourself:
# df[(df["score"] >= 60) and (df["attendance"] >= 80)]

### Example 5 — pandas: `|` (OR) and `~` (NOT)

Which students **need help**, meaning a low score **or** low attendance? And who is everyone else?

In [ ]:
needs_help = (df["score"] < 60) | (df["attendance"] < 70)

print(df[needs_help]["name"].tolist())
print(df[~needs_help]["name"].tolist())

- `|` gives `True` when **at least one** condition is `True`, just like OR on bits.
- `~` **flips** every `True`/`False` in the column, just like NOT flips every bit. `~needs_help` means "everyone who does **not** need help".

In pandas, use `~` and not Python's `not`, which gives the same "truth value is ambiguous" error as `and`.

### Example 6 — pandas: `^` (XOR) means "exactly one"

The teacher wants to remind students who submitted **only one** of the two homeworks: not both, and not neither.

In [ ]:
df[["name", "hw1", "hw2"]]

In [ ]:
only_one = df["hw1"] ^ df["hw2"]
df[only_one]["name"].tolist()

XOR is `True` when the two values are **different**: one `True` and one `False`. That is exactly what "submitted only one" means.

Combining operators works too. Who did homework 1 but **not** homework 2?

In [ ]:
df[df["hw1"] & ~df["hw2"]]["name"].tolist()

### Example 7 — Colors: pack and unpack with shifts `<<` `>>`

Web colors such as `#FF8800` (orange) store **three numbers in one**: red, green and blue, with 8 bits each. Hex is a short way to write those bits, since each hex digit stands for exactly 4 bits:

| | red | green | blue |
|---|---|---|---|
| hex | `FF` (255) | `88` (136) | `00` (0) |
| bit positions | 23 – 16 | 15 – 8 | 7 – 0 |

**How shifts work:**
- `x << n` slides the bits **left** by `n` places and fills the gap with 0s. That **multiplies** by 2ⁿ.
- `x >> n` slides the bits **right** by `n` places, and the rightmost bits **fall off**. That **divides** by 2ⁿ and rounds down.

In [ ]:
show(5, 5 << 1, 5 << 3, 5 >> 1, width=8)

To **unpack** a color, shift the part you want to the right end, then keep only the last 8 bits with the mask `& 0xFF` (`0xFF` = `11111111`):

In [ ]:
orange = 0xFF8800

red = orange >> 16
green = (orange >> 8) & 0xFF
blue = orange & 0xFF
print(red, green, blue)

# And to PACK three numbers back into one color: shift left, then combine with |
packed = (255 << 16) | (136 << 8) | 0
print(hex(packed))

The same pack/unpack idea is used for image pixels, network addresses, file formats and game saves. It lets you fit many small numbers into one big one.

---
## Part 3 — All the Operators in One Place

### 3.1 Bitwise operators

| operator | name | rule | `12 op 10` (`1100` op `1010`) |
|---|---|---|---|
| `&` | AND | 1 if **both** bits are 1 | `1000` = 8 |
| `\|` | OR | 1 if **at least one** bit is 1 | `1110` = 14 |
| `^` | XOR | 1 if the bits are **different** | `0110` = 6 |
| `~` | NOT | flips **all** bits (`~x` = `-x - 1`) | `~12` = -13 |
| `<<` | shift left | push zeros in from the right (×2 each step) | `12 << 1` = 24 |
| `>>` | shift right | the rightmost bits fall off (÷2 each step) | `12 >> 1` = 6 |

**How to work them out by hand:** write the numbers in binary, **one under the other**, and go column by column. There is no carrying, unlike addition.

In [ ]:
a, b = 12, 10
show(a, b)
print('-' * 12)
show(a & b, a | b, a ^ b)

⚠️ **Don't mix these up:**
- `and` / `or` / `not` work on **one** True/False value.
- `&` / `|` / `~` work on **bits** and on **whole pandas columns**.
- `^` is **not** "power" in Python, so `2 ^ 3` is `1`. Power is `**`.

### 3.2 Number systems (from the slides)

| prefix | base | digits | 203 written this way |
|---|---|---|---|
| `0b` | 2 | 0–1 | `0b11001011` |
| `0o` | 8 | 0–7 | `0o313` |
| `0x` | 16 | 0–9, A–F | `0xCB` |

Hex needs 16 digits, so after 9 it uses letters: **A=10, B=11, C=12, D=13, E=14, F=15**.

| hex | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | A | B | C | D | E | F |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| bits | 0000 | 0001 | 0010 | 0011 | 0100 | 0101 | 0110 | 0111 | 1000 | 1001 | 1010 | 1011 | 1100 | 1101 | 1110 | 1111 |

**Shortcut:** one hex digit is exactly **4 bits**, and one octal digit is exactly **3 bits**. So to convert binary to hex, group the bits in fours **from the right**: `1100 1011` → `C B` → `0xCB`. For octal, group in threes: `11 001 011` → `3 1 3` → `0o313`.

**Decimal → binary by hand:** write out the place values `128 64 32 16 8 4 2 1`. Take the **biggest one that fits**, subtract it, and repeat with what's left. Put a `1` under every place value you used:

```
203 − 128 = 75     75 − 64 = 11     11 − 8 = 3     3 − 2 = 1     1 − 1 = 0
used: 128, 64, 8, 2, 1   →   1100 1011
```
(Part 4 shows the other method from the slides, dividing by 2, written as a `while` loop.)

In [ ]:
print(0b11001011, 0o313, 0xCB)
print(bin(203), oct(203), hex(203))

### 3.3 Assignment operators

`x op= y` is shorthand for `x = x op y`. It works for **every** operator. We already used `|=`, `^=` and `&=` in the permission examples.

| shorthand | same as | `x = 5` → |
|---|---|---|
| `x += 3` | `x = x + 3` | 8 |
| `x -= 3` | `x = x - 3` | 2 |
| `x *= 3` | `x = x * 3` | 15 |
| `x /= 3` | `x = x / 3` | 1.666... *(always a float!)* |
| `x //= 3` | `x = x // 3` | 1 |
| `x %= 3` | `x = x % 3` | 2 |
| `x **= 3` | `x = x ** 3` | 125 |
| `x &= 3` | `x = x & 3` | 1 |
| `x \|= 3` | `x = x \| 3` | 7 |
| `x ^= 3` | `x = x ^ 3` | 6 |
| `x >>= 3` | `x = x >> 3` | 0 |
| `x <<= 3` | `x = x << 3` | 40 |

⚠️ `x << 1` on a line by itself **computes** a value but does **not** store it. To change `x` you need `x <<= 1`.

### ✅ Check yourself

Answer on paper, then click to reveal. If you get all six, you're ready for the exercises.

**`9 & 5`, `9 | 5`, `9 ^ 5`?**

<details><summary>Show answer</summary>

`1001` vs `0101` → `&` = `0001` = **1**, `|` = `1101` = **13**, `^` = `1100` = **12**

</details>

**`3 << 4`? `50 >> 3`?**

<details><summary>Show answer</summary>

3 × 16 = **48**. 50 // 8 = **6**

</details>

**`0x3F` in decimal? `77` in binary?**

<details><summary>Show answer</summary>

3×16 + 15 = **63**. 77 = 64 + 8 + 4 + 1 → **`1001101`**

</details>

**`p = 0`, then `p |= 4`, `p |= 1`, `p ^= 4`. What is `p`?**

<details><summary>Show answer</summary>

`000` → `100` → `101` → `001` = **1**

</details>

**Why does `df[df['a'] > 1 and df['b'] < 5]` crash, and what is the fix?**

<details><summary>Show answer</summary>

`and` works on a single True/False, not on a whole column. Fix: **`df[(df['a'] > 1) & (df['b'] < 5)]`**, with parentheses around each condition.

</details>

**`x = 8`, then `x >>= 2`, `x ^= 1`. What is `x`?**

<details><summary>Show answer</summary>

8 >> 2 = 2 (`10`), then `10 ^ 01` = `11` = **3**

</details>

In [ ]:
# Check any of the answers here
print(9 & 5, 9 | 5, 9 ^ 5, 3 << 4, 50 >> 3, 0x3F, bin(77))

---
## Part 4 — Loops: a Few Quick Examples

You already know `for` and `while`. Here they are applied to today's topic. Read each one, predict the output, then run it.

**`for`** = you know **how many times**. **`while`** = repeat **until** something changes.

In [ ]:
# for: print every 3-bit number in binary, from 000 to 111
for n in range(8):
    print(n, '{:03b}'.format(n))

In [ ]:
# while: convert to binary by hand, dividing by 2 until nothing is left
# (we don't know in advance how many steps this takes, so we use while)
n = 203
bits = ""
while n > 0:
    bits = str(n % 2) + bits     # the remainder is the next bit (added on the left)
    n //= 2
print(bits, bin(203))

In [ ]:
# while + <<=: how many doublings does it take to pass 1000?
x = 1
steps = 0
while x <= 1000:
    x <<= 1          # same as x = x * 2
    steps += 1
print(steps, x)

In [ ]:
# end=" " keeps printing on the same line
for i in range(1, 6):
    print('{:02d}'.format(i * 3), end=" ")
print()   # finish the line

---
## Part 5 — Exercises

10 exercises, from easy to medium. For *predict* exercises, write your answer down **before** you run the cell.

### Exercise 1 — Convert by Hand ✍️

On paper, write **45** in binary, hexadecimal and octal.

*Hint: use the place values 32 16 8 4 2 1. For hex, group the bits in fours from the right. For octal, group them in threes.*

Fill in your answers and run the cell. If you are right, all three numbers print `45`.

In [ ]:
print(0b___, 0x___, 0o___)   # replace each ___ with your answer

print(bin(45), hex(45), oct(45))   # Python's answer, to check yourself

### Exercise 2 — Predict: Permissions 🔐

Use the permission flags from Part 2. Predict both lines, then run the cell.

In [ ]:
READ_PERMISSION = 4
WRITE_PERMISSION = 2
EXECUTE_PERMISSION = 1

p = READ_PERMISSION | EXECUTE_PERMISSION
print(p, p & WRITE_PERMISSION, p & READ_PERMISSION)

p ^= READ_PERMISSION
print(p)

### Exercise 3 — Write It: Linux-Style Permissions 🐧

Linux displays permissions as 3 letters: `r` for read, `w` for write and `x` for execute, with a `-` wherever a permission is missing.

Write `describe(p)` that returns this 3-letter string for a permission number `p` from 0 to 7.

- `describe(7)` ➞ `"rwx"`
- `describe(6)` ➞ `"rw-"`
- `describe(5)` ➞ `"r-x"`
- `describe(0)` ➞ `"---"`

*Hint: start with `text = ""`. Then, for each of the three permissions, check it with `&` and add either the letter or `"-"`.*

In [ ]:
READ_PERMISSION = 4
WRITE_PERMISSION = 2
EXECUTE_PERMISSION = 1

# YOUR CODE HERE — define describe(p)


print(describe(7))
print(describe(6))
print(describe(5))
print(describe(0))

### Exercise 4 — Debug: pandas Filter 🐼

We want the students with a score **of at least 70** and attendance **of at least 85**. The expected answer is `['Aziz', 'Eun']`.

The cell below crashes. Fix it. There are **two** things to change, both from the "classic mistakes" in Example 4.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name":       ["Aziz", "Bona", "Chen", "Dilnoza", "Eun", "Farrukh"],
    "score":      [85, 52, 91, 67, 74, 45],
    "attendance": [95, 80, 60, 90, 85, 70],
    "hw1":        [True, True, False, True, False, False],
    "hw2":        [True, False, False, True, True, True],
})

good = df[df["score"] >= 70 and df["attendance"] >= 85]
print(good["name"].tolist())

### Exercise 5 — Write It: pandas Reminder List 📋

Using the same `df` (run Exercise 4's cell first), select every student who **did not submit homework 2**, **or** who scored **below 50**.

**Expected:** `['Bona', 'Chen', 'Farrukh']`

*Hint: you need `~` and `|`. Don't forget the parentheses!*

In [ ]:
# YOUR CODE HERE
reminder = df[___]
print(reminder["name"].tolist())

### Exercise 6 — Spot the Infinite Loop 🔁

**Don't run these!** Exactly **one** of them never stops. Which one, and why?

**A)**
```python
x = 1
while x < 100:
    x <<= 1
```

**B)**
```python
n = 10
while n != 0:
    n -= 3
```

**C)**
```python
for i in range(10):
    i -= 1
```

**D)**
```python
n = 50
while n > 0:
    n >>= 1
```

In [ ]:
# Write your answer as a comment, e.g.  # B, because ...

### Exercise 7 — Puzzle: The XOR Swap 🧩

Predict what prints. Finish the table first: the first two rows are done for you. Remember that `^` gives 1 where the bits are **different**.

| line | `a` | `b` |
|---|---|---|
| start | `0101` (5) | `1001` (9) |
| `a ^= b` | `1100` (12) | `1001` (9) |
| `b ^= a` | | |
| `a ^= b` | | |

In [ ]:
a = 5    # 0101
b = 9    # 1001
a ^= b
b ^= a
a ^= b
print(a, b)

### Exercise 8 — Debug: Building a Color 🎨

`make_color(r, g, b)` should **pack** three numbers into one color, as in Example 7.

**Expected:**
```
0xff8800
0x123456
```
It prints `0xff0000` and `0x120056` instead. Find the bug and fix it.

In [ ]:
def make_color(r, g, b):
    return (r << 16) | (g >> 8) | b

print(hex(make_color(255, 136, 0)))
print(hex(make_color(0x12, 0x34, 0x56)))

### Exercise 9 — Write It: Count the 1 Bits 💡

Write `count_ones(n)` that returns how many `1`s are in the binary form of `n`, **without** using `bin()`. For example, it could count how many permissions a user has.

- `count_ones(13)` ➞ `3` *(13 = `1101`)*
- `count_ones(7)` ➞ `3` *(all three permissions)*
- `count_ones(0)` ➞ `0`
- `count_ones(255)` ➞ `8`

*Hint: `n & 1` gives the last bit (0 or 1). `n >>= 1` drops the last bit. Use a `while` loop and repeat until `n` is 0.*

In [ ]:
# YOUR CODE HERE — define count_ones(n)


print(count_ones(13))
print(count_ones(7))
print(count_ones(0))
print(count_ones(255))
print(count_ones(2026) == bin(2026).count("1"))   # should be True

### Exercise 10 — Mini-Project: Multiplication Table 📋

This is Exercise 6.1 from the slides. Print the full multiplication table for **2 to 9**, laid out like this. Each column is one number from 2 to 9, and each row is one multiplier from 1 to 9:

```
02x01=02 03x01=03 04x01=04 05x01=05 06x01=06 07x01=07 08x01=08 09x01=09
02x02=04 03x02=06 04x02=08 05x02=10 06x02=12 07x02=14 08x02=16 09x02=18
...
02x09=18 03x09=27 04x09=36 05x09=45 06x09=54 07x09=63 08x09=72 09x09=81
```

*Hints:*
- You need **two** loops, one inside the other. Which one is the **outer** loop: the rows (1–9) or the columns (2–9)?
- `'{0:02d}x{1:02d}={2:02d}'.format(3, 4, 12)` gives `'03x04=12'`
- `print(..., end=" ")` stays on the same line. An empty `print()` moves to the next line.

In [ ]:
# YOUR CODE HERE

---

### 🎉 Done!

**Finished early?** Write `has_all(p, needed)`, which returns `True` if permission number `p` includes **every** permission in `needed`. For example, `has_all(7, READ_PERMISSION | WRITE_PERMISSION)` ➞ `True` and `has_all(5, READ_PERMISSION | WRITE_PERMISSION)` ➞ `False`. *(It's one line!)*